# Architecture C



In [ ]:
import os
import sys
import subprocess
import pathlib

REPO_URL = "https://github.com/LLM4SE-group-15/ArchitecturesForCodeDevelopmentWithLLMs.git"
REPO_DIR = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

print(f"Using repo at {REPO_DIR.resolve()}")

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import getpass
from huggingface_hub import login

# Forza sempre l'uso del token che inserisci

os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN (kept hidden): ")

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

os.environ["ARCHITECTURE"] = "C"
print("ARCHITECTURE set to", os.environ["ARCHITECTURE"])

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
try:
    user_info = api.whoami(token=os.environ["HF_TOKEN"])
    print("Logged in to Hugging Face as:", user_info.get("name") or user_info.get("user"))
except Exception as exc:
    print("Login check failed:", exc)

In [ ]:
import json
import time
import logging
import pathlib
import os

from datetime import datetime

DEFAULT_ROOT = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")
ROOT = DEFAULT_ROOT if DEFAULT_ROOT.exists() else pathlib.Path.cwd()

sys.path.insert(0, str(ROOT))

LOG_DIR = ROOT / "log"
LOG_DIR.mkdir(exist_ok=True)

logger = logging.getLogger("architecture_C")
logger.setLevel(logging.INFO)
if logger.handlers:
    logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_DIR / "architecture_C.log")
stream_handler = logging.StreamHandler()
for handler in (file_handler, stream_handler):
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger ready. Repo root: %s", ROOT)
logger.info("Log files: %s", LOG_DIR)
print("Logs ->", LOG_DIR)

In [ ]:
import time
import json
from src.data.task_loader import APPSTaskLoader
from src.graph.graph import run_graph
from src.agents.llm import Architecture

ARCH = Architecture.C
SEED_FIXED = 31

def run_sample_tasks(
    per_level: int = 5,
    split: str = "test",
    difficulties: tuple[str, ...] = ("introductory", "interview", "competition"),
    shuffle: bool = True,
):
    loader = APPSTaskLoader(split=split)
    if shuffle:
        # Nota: APPSTaskLoader potrebbe non esporre direttamente _dataset o shuffle in questo modo
        # Assumiamo che il metodo shuffle funzioni come nello snippet originale dell'utente
        if hasattr(loader, "_dataset"):
             loader._dataset = loader.dataset.shuffle(seed=SEED_FIXED)
             
    tasks = []
    for diff in difficulties:
        tasks.extend(loader.load_by_difficulty(diff, limit=per_level))

    results = []
    total = len(tasks)
    logger.info("Loaded %s tasks (%s per difficulty: %s)", total, per_level, ", ".join(difficulties))
    print(f"Starting benchmark on {total} tasks...")
    
    for idx, task in enumerate(tasks, 1):
        logger.info("Running %s/%s %s (%s)", idx, total, task.task_id, task.difficulty)
        # Stampiamo a video solo una riga di 'status' per non intasare l'output
        print(f"[{idx}/{total}] Task {task.task_id} ({task.difficulty})... ", end="", flush=True)
        
        start = time.time()
        # Eseguiamo il grafo per il task corrente
        state = run_graph(
            task_id=task.task_id,
            task_description=task.question,
            test_inputs=task.inputs,
            test_outputs=task.outputs,
            architecture=ARCH,
        )

        elapsed = time.time() - start
        
        # Prepariamo il record
        record = {
            "task_id": task.task_id,
            "difficulty": task.difficulty,
            "architecture": str(ARCH.value),
            "test_passed": state["test_passed"],
            "developer_tier": state.get("developer_tier"),
            "escalations": state["escalations"],
            "story_points_initial": state.get("story_points_initial"),
            "story_points_final": state.get("story_points_current"),
            "elapsed_seconds": elapsed,
        }
        results.append(record)
        
        # Logghiamo info e completiamo la riga di output a video
        logger.info(
            "Finished %s | pass=%s tier=%s escalations=%s elapsed=%.1fs",
            task.task_id,
            state["test_passed"],
            record["developer_tier"],
            record["escalations"],
            elapsed,
        )
        status_str = "PASS" if state["test_passed"] else "FAIL"
        print(f"{status_str} in {elapsed:.1f}s")

        # Scriviamo nel file jsonl
        with open(LOG_DIR / "architecture_C.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
            
    return results

# Esecuzione
sample_results = run_sample_tasks(per_level=5, difficulties=("introductory", "interview", "competition"), shuffle=True)

# Visualizziamo solo un sommario finale dei risultati invece dell'intera lista
passed_count = sum(1 for r in sample_results if r['test_passed'])
print(f"\nBenchmark Completed. Passed: {passed_count}/{len(sample_results)}")

In [ ]:
!cd log && cat architecture_C.jsonl

## Evaluation Metrics for Architecture C

This section calculates the evaluation metrics as specified in `evaluation.md`:

- **Primary Metrics**: Pass Rate, Pass@1
- **Cost Metrics**: Execution Time, API Calls
- **Pass Rate by Difficulty**: Introductory, Interview, Competition

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Load results from JSONL file
log_file = LOG_DIR / "architecture_C.jsonl"
results = []
if log_file.exists():
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                results.append(json.loads(line))
else:
    print(f"Warning: {log_file} not found. Make sure to run the tasks first.")

df = pd.DataFrame(results)
print(f"Loaded {len(df)} task results")

# Mostra tutte le righe
pd.set_option('display.max_rows', None)
df

In [ ]:
# Helper function for Wilson score confidence interval
def wilson_ci(successes, n, confidence=0.95):
    """Calculate Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0, 0, 0
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p_hat = successes / n
    denominator = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denominator
    margin = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * n)) / n) / denominator
    return p_hat, center - margin, center + margin

# Calculate Pass Rate
total_tasks = len(df)
passed_tasks = df["test_passed"].sum()
pass_rate = passed_tasks / total_tasks if total_tasks > 0 else 0

# For Architecture B, Pass@1 is the same as Pass Rate (no retries)
pass_at_1 = pass_rate

# Calculate 95% Confidence Interval
p, ci_low, ci_high = wilson_ci(passed_tasks, total_tasks)

print("=" * 60)
print("PRIMARY METRICS - Architecture C")
print("=" * 60)
print(f"\nTotal Tasks Evaluated: {total_tasks}")
print(f"Tasks Passed: {passed_tasks}")
print(f"Pass Rate: {pass_rate:.2%}")
print(f"Pass@1: {pass_at_1:.2%}")
print(f"95% Confidence Interval: [{ci_low:.2%}, {ci_high:.2%}]")
print("=" * 60)

In [ ]:
# Calculate Pass Rate by Difficulty
difficulty_stats = df.groupby("difficulty").agg(
    total=pd.NamedAgg(column="test_passed", aggfunc="count"),
    passed=pd.NamedAgg(column="test_passed", aggfunc="sum")
)
difficulty_stats["pass_rate"] = difficulty_stats["passed"] / difficulty_stats["total"]

# Add 95% CI for each difficulty
cis = []
for _, row in difficulty_stats.iterrows():
    _, ci_l, ci_h = wilson_ci(int(row["passed"]), int(row["total"]))
    cis.append(f"[{ci_l:.2%}, {ci_h:.2%}]")
difficulty_stats["95% CI"] = cis

print("\n" + "=" * 60)
print("PASS RATE BY DIFFICULTY")
print("=" * 60)
print("\n| Difficulty    | Total | Passed | Pass Rate | 95% CI |")
print("|---------------|-------|--------|-----------|--------|")
for diff in ["introductory", "interview", "competition"]:
    if diff in difficulty_stats.index:
        row = difficulty_stats.loc[diff]
        print(f"| {diff:<13} | {int(row['total']):>5} | {int(row['passed']):>6} | {row['pass_rate']:>8.2%} | {row['95% CI']} |")
print("=" * 60)

In [ ]:
# Calculate Cost Metrics
avg_time = df["elapsed_seconds"].mean()
std_time = df["elapsed_seconds"].std()
median_time = df["elapsed_seconds"].median()
total_time = df["elapsed_seconds"].sum()
min_time = df["elapsed_seconds"].min()
max_time = df["elapsed_seconds"].max()

# For Architecture A, each task is 1 API call (single-agent, no retry)
api_calls_per_task = 1
total_api_calls = total_tasks * api_calls_per_task

print("\n" + "=" * 60)
print("COST METRICS")
print("=" * 60)
print(f"\nExecution Time Statistics:")
print(f"  Average Time per Task: {avg_time:.2f} seconds")
print(f"  Std Dev: {std_time:.2f} seconds")
print(f"  Median Time: {median_time:.2f} seconds")
print(f"  Min Time: {min_time:.2f} seconds")
print(f"  Max Time: {max_time:.2f} seconds")
print(f"  Total Execution Time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"\nAPI Calls:")
print(f"  API Calls per Task: {api_calls_per_task}")
print(f"  Total API Calls: {total_api_calls}")
print("=" * 60)

In [ ]:
# Execution Time by Difficulty
time_by_diff = df.groupby("difficulty")["elapsed_seconds"].agg(["mean", "std", "median"])

print("\n" + "=" * 60)
print("EXECUTION TIME BY DIFFICULTY")
print("=" * 60)
print("\n| Difficulty    | Avg Time (s) | Std Dev (s) | Median (s) |")
print("|---------------|--------------|-------------|------------|")
for diff in ["introductory", "interview", "competition"]:
    if diff in time_by_diff.index:
        row = time_by_diff.loc[diff]
        print(f"| {diff:<13} | {row['mean']:>12.2f} | {row['std']:>11.2f} | {row['median']:>10.2f} |")
print("=" * 60)

In [ ]:
# For Architecture A, there are no retries (single-shot)
print("\n" + "=" * 60)
print("RETRY DYNAMICS")
print("=" * 60)
print("\nNote: Architecture A is a single-shot baseline with no retry mechanism.")
print("\n| Retries | Count | Percentage |")
print("|---------|-------|------------|")
print(f"| 0       | {total_tasks:>5} | {100:>10.2f}% |")
print("=" * 60)

In [ ]:
# Visualizations
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

# Create a figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Architecture A - Evaluation Metrics", fontsize=16, fontweight="bold")

# 1. Pass Rate Bar Chart
ax1 = axes[0, 0]
pass_rate_vals = [difficulty_stats.loc[diff, "pass_rate"] * 100 if diff in difficulty_stats.index else 0 
              for diff in ["introductory", "interview", "competition"]]
colors = ["#2ecc71" if r > 50 else "#e74c3c" for r in pass_rate_vals]
bars = ax1.bar(["Introductory", "Interview", "Competition"], pass_rate_vals, color=colors, edgecolor="black")
ax1.set_ylabel("Pass Rate (%)")
ax1.set_title("Pass Rate by Difficulty")
ax1.set_ylim(0, 100)
for bar, rate in zip(bars, pass_rate_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f"{rate:.1f}%", 
             ha="center", fontsize=11, fontweight="bold")

# 2. Execution Time Box Plot by Difficulty
ax2 = axes[0, 1]
difficulty_order = ["introductory", "interview", "competition"]
df_ordered = df[df["difficulty"].isin(difficulty_order)].copy()
df_ordered["difficulty"] = pd.Categorical(df_ordered["difficulty"], categories=difficulty_order, ordered=True)
sns.boxplot(data=df_ordered, x="difficulty", y="elapsed_seconds", ax=ax2, palette="viridis")
ax2.set_xlabel("Difficulty")
ax2.set_ylabel("Execution Time (seconds)")
ax2.set_title("Execution Time Distribution by Difficulty")

# 3. Pass/Fail Pie Chart
ax3 = axes[1, 0]
labels = ["Passed", "Failed"]
sizes = [passed_tasks, total_tasks - passed_tasks]
colors_pie = ["#2ecc71", "#e74c3c"]
explode = (0.05, 0)
ax3.pie(sizes, explode=explode, labels=labels, colors=colors_pie, autopct="%1.1f%%",
        shadow=True, startangle=90)
ax3.set_title("Overall Pass/Fail Distribution")

# 4. Execution Time Histogram
ax4 = axes[1, 1]
ax4.hist(df["elapsed_seconds"], bins=15, edgecolor="black", color="#3498db", alpha=0.7)
ax4.axvline(avg_time, color="red", linestyle="--", linewidth=2, label=f"Mean: {avg_time:.2f}s")
ax4.axvline(median_time, color="green", linestyle="--", linewidth=2, label=f"Median: {median_time:.2f}s")
ax4.set_xlabel("Execution Time (seconds)")
ax4.set_ylabel("Frequency")
ax4.set_title("Execution Time Distribution")
ax4.legend()

plt.tight_layout()
plt.savefig(LOG_DIR / "architecture_C_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nVisualization saved to: {LOG_DIR / 'architecture_A_metrics.png'}")

In [ ]:
# Summary Table
summary_data = {
    "Metric": [
        "Architecture",
        "Total Tasks",
        "Passed Tasks",
        "Pass Rate",
        "Pass@1",
        "95% CI (Lower)",
        "95% CI (Upper)",
        "Avg Execution Time (s)",
        "Std Execution Time (s)",
        "Median Execution Time (s)",
        "Total Execution Time (s)",
        "API Calls per Task",
        "Total API Calls",
        "Retry Count",
        "Escalations"
    ],
    "Value": [
        "A (Single-Agent Baseline)",
        total_tasks,
        passed_tasks,
        f"{pass_rate:.4f}",
        f"{pass_at_1:.4f}",
        f"{ci_low:.4f}",
        f"{ci_high:.4f}",
        f"{avg_time:.2f}",
        f"{std_time:.2f}",
        f"{median_time:.2f}",
        f"{total_time:.2f}",
        api_calls_per_task,
        total_api_calls,
        0,
        0
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 60)
print("SUMMARY TABLE FOR ARCHITECTURE C")
print("=" * 60)
print(summary_df.to_string(index=False))
summary_df.to_csv(LOG_DIR / "architecture_A_summary.csv", index=False)

# Save difficulty breakdown
difficulty_summary = difficulty_stats.reset_index()
difficulty_summary.columns = ["Difficulty", "Total", "Passed", "Pass Rate", "95% CI"]
difficulty_summary.to_csv(LOG_DIR / "architecture_C_by_difficulty.csv", index=False)
print(f"\nAll results saved to {LOG_DIR}")